In [ ]:
import os, time, random
import cdsapi
import xarray as xr

NETID = "k16v981"
BASE = f"/home/{NETID}/my_work/data/era5/era5_sst"
OUT_DIR = BASE
TMP_DIR = os.path.join(OUT_DIR, "_tmp_monthly_sst")
os.makedirs(TMP_DIR, exist_ok=True)

START_YEAR = 2021
END_YEAR   = 2025

MONTHS = [f"{m:02d}" for m in range(1, 13)]

MAX_RETRIES  = 6
BACKOFF_BASE = 2.0
JITTER       = (0.0, 1.0)

c = cdsapi.Client()

def dl_month(year: int, month: str) -> str:
    """Download one month of ERA5 global monthly mean SST as NetCDF."""
    out_path = os.path.join(TMP_DIR, f"era5_sst_monthly_global_{year}_{month}.nc")
    if os.path.exists(out_path):
        print(f"✅ Exists: {os.path.basename(out_path)}")
        return out_path

    req = {
        "product_type": "monthly_averaged_reanalysis",
        "variable": ["sea_surface_temperature"],
        "year": str(year),
        "month": month,
        "time": "00:00",
        "format": "netcdf",
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"📥 Download {year}-{month} (attempt {attempt}) ...")
            c.retrieve("reanalysis-era5-single-levels-monthly-means", req, out_path)
            print(f"💾 Saved: {os.path.basename(out_path)}")
            return out_path
        except Exception as e:
            print(f"❌ Failed {year}-{month} (attempt {attempt}): {e}")
            sleep_s = BACKOFF_BASE * attempt + random.uniform(*JITTER)
            print(f"⏳ Backing off {sleep_s:.1f}s")
            time.sleep(sleep_s)

    raise RuntimeError(f"Gave up downloading {year}-{month}")

def build_year(year: int):
    """Concat 12 monthly files into one yearly file and delete monthly pieces."""
    year_out = os.path.join(OUT_DIR, f"era5_sst_{year}.nc")
    if os.path.exists(year_out):
        print(f"✅ Year exists: {os.path.basename(year_out)} (skipping)")
        return

    paths = []
    for m in MONTHS:
        paths.append(dl_month(year, m))
        time.sleep(1 + random.uniform(0, 1.5))

    # Open + concat along time (monthly)
    # Use combine="by_coords" to avoid manual concat if metadata is nice
    ds = xr.open_mfdataset(paths, combine="by_coords")

    # Optional: ensure variable name matches your older files
    # Some files use "sst" but ERA5 uses "sst" or "sea_surface_temperature" depending on source.
    # We'll keep whatever it is, but you can normalize:
    if "sea_surface_temperature" in ds.data_vars and "sst" not in ds.data_vars:
        ds = ds.rename({"sea_surface_temperature": "sst"})

    # Write yearly file
    ds.to_netcdf(year_out)
    ds.close()
    print(f"✅ Wrote: {os.path.basename(year_out)}")

    # Delete monthly files
    for p in paths:
        try:
            os.remove(p)
        except FileNotFoundError:
            pass
    print(f"🧹 Cleaned monthly pieces for {year}")

def main():
    for y in range(START_YEAR, END_YEAR + 1):
        build_year(y)
    # Optional: remove tmp dir if empty
    try:
        if len(os.listdir(TMP_DIR)) == 0:
            os.rmdir(TMP_DIR)
    except Exception:
        pass
    print("✅ Done")

if __name__ == "__main__":
    main()

📥 Download 2021-01 (attempt 1) ...


2026-01-28 15:50:19,188 INFO Request ID is 0ae6c1d5-0416-41ef-8931-8d7bb381e892
2026-01-28 15:50:19,366 INFO status has been updated to accepted
2026-01-28 15:50:32,396 INFO status has been updated to successful


d23a589bb8fc4907d39a55bd0b117467.nc:   0%|          | 0.00/980k [00:00<?, ?B/s]

💾 Saved: era5_sst_monthly_global_2021_01.nc
📥 Download 2021-02 (attempt 1) ...


2026-01-28 15:50:37,448 INFO Request ID is 7e96b4fb-6e93-49ca-b18c-9247ee5d6649
2026-01-28 15:50:37,622 INFO status has been updated to accepted
2026-01-28 15:50:50,736 INFO status has been updated to running
2026-01-28 15:50:58,508 INFO status has been updated to successful


...

💾 Saved: era5_sst_monthly_global_2025_07.nc
📥 Download 2025-08 (attempt 1) ...


2026-01-28 16:13:04,188 INFO Request ID is 7fb35a20-ef14-432a-bc77-8c31893bd296
2026-01-28 16:13:04,362 INFO status has been updated to accepted
2026-01-28 16:13:17,297 INFO status has been updated to running
2026-01-28 16:13:25,064 INFO status has been updated to successful


b98a83785737133d777421a6e46190b6.nc:   0%|          | 0.00/973k [00:00<?, ?B/s]

💾 Saved: era5_sst_monthly_global_2025_08.nc
📥 Download 2025-09 (attempt 1) ...


2026-01-28 16:13:29,784 INFO Request ID is 492f59bf-c813-4d0e-8f30-aa59205efd5e
2026-01-28 16:13:29,952 INFO status has been updated to accepted
